# Distribution Analysis — GDP per Capita & Unemployment Rate (2025)

Two histograms showing the **distribution** of values across all OECD countries for 2025.

A histogram divides values into intervals (bins) and counts how many countries fall in each interval. This reveals:
- **Central tendency** — where most countries cluster
- **Dispersion** — how spread out the values are
- **Skewness** — whether the distribution is symmetric or pulls toward one extreme
- **Outliers** — countries with unusually high or low values

Data source: `datasets/Per-Year/gdp_2025.xlsx` and `datasets/Per-Year/unemployment_2025.xlsx`

In [16]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

# ── Load and clean ────────────────────────────────────────────────────────────
gdp_all = pd.read_excel('datasets/Per-Year/gdp_2025.xlsx')
une_all = pd.read_excel('datasets/Per-Year/unemployment_2025.xlsx')

AGGREGATE_AREAS = {'EA', 'EU27_2020', 'G7', 'OECD', 'OECDE', 'USMCA'}
gdp_all = gdp_all[~gdp_all['REF_AREA'].isin(AGGREGATE_AREAS)]
une_all = une_all[~une_all['REF_AREA'].isin(AGGREGATE_AREAS)]

df = gdp_all.merge(une_all, on='REF_AREA', how='inner')

# ── Bin countries by unemployment rate (X axis) ────────────────────────────
n_bins = 8
df['UNE_BIN'] = pd.cut(df['UNE_RATE_PCT_AVG_2025'], bins=n_bins)
binned = (
    df.groupby('UNE_BIN', observed=True)
    .agg(
        GDP_MEAN=('GDP_PER_CAPITA_USD_PPP_AVG_2025', 'mean'),
        COUNT=('REF_AREA', 'count'),
        COUNTRIES=('REF_AREA', lambda x: ', '.join(sorted(x)))
    )
    .reset_index()
)
binned['BIN_LABEL'] = binned['UNE_BIN'].apply(lambda b: f'{b.left:.1f}%–{b.right:.1f}%')
binned['BIN_MID']   = binned['UNE_BIN'].apply(lambda b: (b.left + b.right) / 2)

# ── Single histogram: X = unemployment bin, Y = avg GDP per capita ────────
fig = go.Figure(go.Bar(
    x=binned['BIN_LABEL'],
    y=binned['GDP_MEAN'],
    marker_color=[
        '#2ca02c' if m < 4 else ('#ff7f0e' if m < 7 else '#d62728')
        for m in binned['BIN_MID']
    ],
    marker_line=dict(color='white', width=1),
    text=[f'${v:,.0f}<br>({c} {"country" if c==1 else "countries"})' for v, c in zip(binned['GDP_MEAN'], binned['COUNT'])],
    textposition='outside',
    textfont=dict(size=10),
    customdata=binned['COUNTRIES'],
    hovertemplate='Unemployment: %{x}<br>Avg GDP: $%{y:,.0f}<br>Countries: %{customdata}<extra></extra>',
))

fig.update_layout(
    title='Histogram — Average GDP per Capita by Unemployment Rate Bin<br>OECD Countries (2025)',
    xaxis_title='Unemployment Rate (% of labour force)  →  horizontal',
    yaxis_title='Average GDP per Capita (USD PPP)  →  vertical',
    height=540,
    plot_bgcolor='white',
    margin=dict(t=80, b=80),
    bargap=0.08,
)
fig.update_xaxes(gridcolor='#e0e0e0')
fig.update_yaxes(gridcolor='#e0e0e0')

fig.show()